# 🧠 Chapter 03-01a: LangChain Fundamentals — The Framework That Powers GenAI Apps

---

## The Story: From Raw API Calls to Real Applications 🏗️

So far, you've been making direct API calls to OpenAI. That's like building a house with just a hammer — possible, but painful.

**LangChain** is your **full toolkit**. It gives you power tools, pre-cut lumber, and blueprints. It abstracts the tedious parts so you can focus on building amazing AI applications.

> *"LangChain is to GenAI what Django is to web development — the framework that turns experiments into products."*

---

## 🎯 What You'll Learn

| Topic | Why It Matters |
|-------|---------------|
| LangChain Architecture | Understand the pieces and how they fit |
| Chat Models | Unified interface for OpenAI, Anthropic, Google, Ollama |
| Prompt Templates | Reusable, composable prompts |
| Message Types | System, Human, AI — the conversation building blocks |
| Simple Chains | Your first prompt → model → output pipeline |

---

## 🏗️ LangChain Architecture (2024-2026)

```
╔═══════════════════════════════════════════════════════════════════════╗
║              LANGCHAIN PACKAGE STRUCTURE (v0.3+)                     ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  langchain-core          ← The foundation. Interfaces & LCEL.        ║
║    │                        Rarely changes. Stable.                   ║
║    │                                                                  ║
║  langchain-openai        ← OpenAI models, embeddings                 ║
║  langchain-anthropic     ← Anthropic Claude models                   ║
║  langchain-google-genai  ← Google Gemini models                      ║
║  langchain-community     ← 100+ integrations (vector stores, etc.)   ║
║    │                                                                  ║
║  langchain               ← High-level chains, agents. Depends on    ║
║    │                        core + pick your providers               ║
║    │                                                                  ║
║  langgraph               ← Stateful, multi-step agent framework     ║
║                             The future of LangChain agents           ║
║                                                                       ║
║  langsmith               ← Observability, tracing, debugging        ║
║                             Production monitoring                    ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### Key Insight: Modular Design

You don't install everything. You install only what you need:

```python
pip install langchain-core         # Always needed
pip install langchain-openai       # If using OpenAI
pip install langchain-anthropic    # If using Claude
pip install langchain              # For chains, agents
pip install langgraph              # For stateful agents (Ch. 7)
```

---

## 💬 Chat Models: The Core Abstraction

```
╔═══════════════════════════════════════════════════════════════════════╗
║                   CHAT MODELS                                         ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  The most important abstraction in LangChain.                        ║
║  All LLMs are wrapped in a UNIFIED INTERFACE:                        ║
║                                                                       ║
║    input:  List of Messages  →  Chat Model  →  output: AIMessage    ║
║                                                                       ║
║  Supported providers (same interface!):                              ║
║    • ChatOpenAI        (GPT-4o, GPT-4o-mini)                        ║
║    • ChatAnthropic     (Claude 3.5, Claude 4)                       ║
║    • ChatGoogleGenAI   (Gemini Pro, Ultra)                          ║
║    • ChatOllama        (Llama 3, Mistral — local!)                  ║
║                                                                       ║
║  Why this matters:                                                    ║
║    Switch from GPT-4o to Claude = change ONE LINE of code.          ║
║    Your entire app, chains, and prompts stay the same!              ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### The Unified Interface

```python
# These ALL work the same way:
llm = ChatOpenAI(model="gpt-4o-mini")
llm = ChatAnthropic(model="claude-3-5-sonnet")
llm = ChatGoogleGenAI(model="gemini-pro")
llm = ChatOllama(model="llama3")       # FREE, runs locally!

# Same code works for ALL of them:
result = llm.invoke("What is Python?")
print(result.content)
```

---

## 📨 Message Types: The Conversation Protocol

```
╔═══════════════════════════════════════════════════════════════════════╗
║                   MESSAGE TYPES                                       ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  SystemMessage   → Sets the LLM's behavior/persona                  ║
║                     "You are a helpful coding tutor."               ║
║                     Only seen by the model, not the user.           ║
║                                                                       ║
║  HumanMessage    → The user's input/question                        ║
║                     "How do I sort a list in Python?"               ║
║                                                                       ║
║  AIMessage       → The model's response                             ║
║                     "You can use sorted() or list.sort()..."        ║
║                     Returned by llm.invoke()                        ║
║                                                                       ║
║  ToolMessage     → Result from a tool call (Ch. 6)                  ║
║                     Used in agent workflows                          ║
║                                                                       ║
║  Conversation = [SystemMessage, HumanMessage, AIMessage, ...]       ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### Tuple Shorthand

LangChain also supports a convenient tuple syntax:

```python
# These are equivalent:
messages = [SystemMessage(content="You are helpful."), HumanMessage(content="Hi")]
messages = [("system", "You are helpful."), ("human", "Hi")]  # ← Shorter!
```

---

## 📝 Prompt Templates: The Right Way to Build Prompts

```
╔═══════════════════════════════════════════════════════════════════════╗
║              WHY PROMPT TEMPLATES?                                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  ❌ f-string:   f"Translate {text} to {lang}"                        ║
║     Problems:   No validation, not composable, no type safety       ║
║                                                                       ║
║  ✅ Template:   ChatPromptTemplate.from_messages([                   ║
║                   ("system", "You are a translator."),              ║
║                   ("human", "Translate to {language}: {text}")      ║
║                 ])                                                    ║
║                                                                       ║
║  Benefits:                                                            ║
║    • Composable with LCEL (the | operator)                           ║
║    • Version controllable                                            ║
║    • Input validation                                                ║
║    • Reusable across your codebase                                   ║
║    • Can include few-shot examples                                   ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### Template Variable Syntax

```python
# Single braces = variable: {name}
# Double braces = literal brace: {{ not a variable }}

template = ChatPromptTemplate.from_messages([
    ("system", "Return JSON format: {{\"key\": \"value\"}}"),  # Literal braces
    ("human", "Analyze: {text}")                                  # Variable
])
```

---

## 🔗 Simple Chains: prompt | model

```
╔═══════════════════════════════════════════════════════════════════════╗
║            THE LCEL CHAIN PATTERN                                     ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  prompt | model                                                       ║
║    │       │                                                          ║
║    │       └── Any Chat Model (OpenAI, Anthropic, etc.)              ║
║    └────────── ChatPromptTemplate with variables                     ║
║                                                                       ║
║  How it works:                                                        ║
║    1. You call chain.invoke({"var": "value"})                       ║
║    2. Template fills in variables → produces Messages                ║
║    3. Messages pass to the model → produces AIMessage                ║
║    4. You get back the AIMessage                                     ║
║                                                                       ║
║  The | is Python's __or__ operator, overloaded by LangChain.        ║
║  It creates a RunnableSequence under the hood.                       ║
║                                                                       ║
║  You can chain MORE steps:                                           ║
║    prompt | model | output_parser                                    ║
║    prompt | model | StrOutputParser()  ← gets just the text!        ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🗺️ LangChain Mental Model

```
  Your App Code
       │
       ▼
  ┌─────────────────────────────────────────────┐
  │  ChatPromptTemplate                          │
  │    • System message (role/instructions)      │
  │    • Human message (with {variables})        │
  │    • Few-shot examples (optional)            │
  └──────────────┬──────────────────────────────┘
                 │  List[Messages]
                 ▼
  ┌─────────────────────────────────────────────┐
  │  Chat Model (ChatOpenAI, ChatAnthropic...)   │
  │    • Sends to API                            │
  │    • Handles retries, rate limits             │
  │    • Returns AIMessage                        │
  └──────────────┬──────────────────────────────┘
                 │  AIMessage
                 ▼
  ┌─────────────────────────────────────────────┐
  │  Output Parser (optional)                    │
  │    • StrOutputParser → plain text            │
  │    • JsonOutputParser → dict                 │
  │    • PydanticOutputParser → typed object     │
  └──────────────┬──────────────────────────────┘
                 │  Parsed result
                 ▼
  Your App Code uses the result
```

---

## 💡 Interview Prep: Common Questions

```
╔═══════════════════════════════════════════════════════════════════════╗
║           INTERVIEW QUESTIONS FOR THIS MODULE                        ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Q: What is LangChain and why would you use it?                      ║
║  A: A Python framework for building LLM-powered applications.       ║
║     Provides unified interfaces for models, prompt templates,       ║
║     chains, agents, and integrations. Key benefit: swap models      ║
║     (OpenAI → Claude) with one line change. Handles composability   ║
║     via LCEL, the pipe operator for chaining components.            ║
║                                                                       ║
║  Q: What is LCEL?                                                    ║
║  A: LangChain Expression Language. Uses the | operator to chain    ║
║     components: prompt | model | parser. Each component is a       ║
║     "Runnable" with invoke(), stream(), batch() methods.           ║
║     Output of one step feeds into the next. Built-in streaming,    ║
║     batching, and async support.                                     ║
║                                                                       ║
║  Q: What are the different message types in LangChain?              ║
║  A: SystemMessage (sets behavior), HumanMessage (user input),      ║
║     AIMessage (model response), ToolMessage (tool results).        ║
║     These map to the chat completion API's role field.              ║
║                                                                       ║
║  Q: How would you switch from OpenAI to Anthropic in LangChain?    ║
║  A: Change ChatOpenAI() to ChatAnthropic(). That's it. The rest    ║
║     of your code (prompts, chains, parsers) stays exactly the      ║
║     same because they all use the same BaseLanguageModel interface. ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Quick Quiz

---

### Question 1:
What does `prompt | model` create in LangChain?

- A) A string concatenation
- B) A RunnableSequence (chain)
- C) A database query
- D) A parallel execution

<details><summary>Answer</summary>

**B) A RunnableSequence (chain)**

The `|` operator in LCEL creates a RunnableSequence where the output of the left component feeds into the input of the right component. When you call `.invoke()`, the data flows through the entire chain sequentially.

</details>

---

### Question 2:
Which package should you install to use OpenAI models with LangChain?

- A) `pip install langchain` only
- B) `pip install langchain-core langchain-openai`
- C) `pip install openai` only
- D) `pip install langchain-community`

<details><summary>Answer</summary>

**B) `pip install langchain-core langchain-openai`**

LangChain v0.3+ uses modular packages. `langchain-core` provides the base interfaces and LCEL. `langchain-openai` provides ChatOpenAI and OpenAIEmbeddings. You install only what you need.

</details>

---

### Question 3:
What's the purpose of a SystemMessage?

- A) To log errors
- B) To set the model's behavior, persona, and constraints
- C) To store conversation history
- D) To define the output format

<details><summary>Answer</summary>

**B) To set the model's behavior, persona, and constraints**

SystemMessage is the "backstage instructions" for the LLM. It defines who the model should act as, what rules to follow, and any constraints. It persists across the conversation but is invisible to end users in a chat UI.

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 LangChain = modular framework: core + provider packages.      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Chat Models wrap all providers in ONE interface.               │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Messages: System (behavior), Human (input), AI (response).    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 ChatPromptTemplate = reusable prompts with {variables}.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LCEL: prompt | model | parser = composable chain.             │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Swap models by changing ONE line. Everything else stays.       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Hands-On LangChain!

We'll build real chains, test different models, and create a translation pipeline with LangChain. Code time!

---

### 🎉 You Now Understand the LangChain Framework!
Architecture, Chat Models, Messages, Templates, Chains — the building blocks are in place! 🚀